# Étape 4 — HMM Market Regime Detection (v2)
## MASI Hybrid Forecasting System

**v2 — comparative observation study.** v1 (`[log_return, garch_vol]`) gave persistent but *descriptive-only* volatility regimes. v2 tests **5 observation sets** and selects the best on TRAIN+VAL (TEST held out).

The regime feature is decoded with a **causal forward-filter** (leakage-free) — Viterbi smoothing would leak future data.

**Result:** winner = **S3 (momentum `roll_mean_21` + `garch_vol`)** → directionally-consistent regimes, verdict **USABLE**. Honest caveats in the report `etape4_regime_report.md` §6 (the gain is momentum continuation; Étape 5 ablation still required).

## 1. Run the comparative HMM pipeline

In [ ]:
import sys, os, json
sys.path.insert(0, os.path.abspath('.'))

import pandas as pd
from IPython.display import Image, display

import importlib.util as _ilu, sys as _sys; _spec = _ilu.spec_from_file_location('e4', '../scripts/04_hmm_regimes.py'); e4 = _ilu.module_from_spec(_spec); _sys.modules['e4'] = e4; _spec.loader.exec_module(e4)
e4.main()

## 2. Comparative study — 5 observation specs

In [ ]:
comp = json.load(open(os.path.join(e4.REG_DIR, 'spec_comparison.json')))
print('WINNER:', comp['winner'], '|', comp['selection_reason'])
rows = []
for s, d in comp['per_spec_detail'].items():
    bb = d['bull_minus_bear']
    rows.append({'spec': s, 'obs': '+'.join(d['obs_cols']),
                 'persistence': round(d['mean_persistence'], 3), 'BIC': round(d['bic'], 0),
                 'spread_train': round(bb['train'], 6), 'spread_val': round(bb['val'], 6),
                 'spread_test': round(bb['test'], 6)})
pd.DataFrame(rows).set_index('spec')

## 3. Winner regime structure (S3 — momentum + volatility)

In [ ]:
ev = json.load(open(os.path.join(e4.REG_DIR, 'regime_evaluation.json')))
rows = []
for nm, d in ev['per_regime'].items():
    rows.append({'regime': nm, 'share_train': round(d['share_train'], 3),
                 'mean_return_ann': round(d['mean_return_ann'], 4),
                 'vol_ann': round(d['vol_ann'], 4),
                 'duration_days': round(d['expected_duration_days'], 1)})
print('Regimes now have a real return ordering Bear < Neutral < Bull:')
display(pd.DataFrame(rows).set_index('regime'))
print('\nTransition matrix (mean persistence = %.3f):' % ev['mean_persistence'])
display(pd.DataFrame(ev['transmat'], index=ev['regime_names'], columns=ev['regime_names']).round(3))

## 4. Does the regime predict next-day return?

In [ ]:
cond = ev['regime_conditional_next_return']
rows = []
for split in ('train', 'val', 'test'):
    c = cond[split]
    rows.append({'split': split,
                 **{nm: round(c[nm]['mean_next_return'], 6) for nm in ev['regime_names']},
                 'Bull_minus_Bear': round(c['bull_minus_bear'], 6)})
print('Bull-minus-Bear positive on ALL splits -> consistent directional signal:')
display(pd.DataFrame(rows).set_index('split'))
strat = pd.DataFrame(ev['regime_strategy']).T[['sharpe', 'max_drawdown', 'n_trades']]
print('\nRegime-timed strategy (long Bull / short Bear) — positive on all 3 windows:')
strat.round(3)

## 5. Verdict

In [ ]:
v = ev['verdict']
for k, val in v.items():
    print(f'  {k:20s}: {val}')
print('\n  -> USABLE: directionally-consistent regimes.')
print('  -> CAVEAT: the gain is momentum continuation; roll_mean_21 is already a')
print('     CNN-LSTM feature. Étape 5 ablation must confirm the regime adds value.')

## 6. Regime timeline (MASI shaded by causal regime)

In [ ]:
Image(os.path.join(e4.PLOTS_DIR, 'etape4_regime_timeline.png'))

## 7. Spec comparison

In [ ]:
Image(os.path.join(e4.PLOTS_DIR, 'etape4_spec_comparison.png'))

## 8. Transition matrix

In [ ]:
Image(os.path.join(e4.PLOTS_DIR, 'etape4_transition_matrix.png'))

## 9. Regime characteristics

In [ ]:
Image(os.path.join(e4.PLOTS_DIR, 'etape4_regime_characteristics.png'))

## 10. n_states selection & coverage

In [ ]:
Image(os.path.join(e4.PLOTS_DIR, 'etape4_model_selection.png'))

## 11. Takeaways for Étape 5 (CNN-LSTM)

| Insight | Implication |
|---------|-------------|
| Winner = momentum HMM (S3); raw-return HMM (S2) fails OOS | Regime needs a persistent signal, not noisy daily return |
| Bull−Bear next-day spread positive on train/val/test | Directionally consistent — verdict USABLE |
| Regime strategy Sharpe +0.69/+0.93/+0.93 | Robust across all windows |
| Regime ≈ smoothed `roll_mean_21`, already a CNN-LSTM feature | Marginal value uncertain — **ablation required** |
| RULE 6/8 | If the with/without-regime ablation shows no gain, drop the regime |